# 組合せベイズ最適化：BOCS・COMBO・Hamming探索

小さな二値空間で、モデル学習から候補提案、目的関数評価、観測更新までを実行します。`fit_model=False` は学習を省略した動作確認用であり、ここでは通常の `fit_model=True` を使用します。

In [ ]:
import torch
from robotorchan.models.standard import BOCSSurrogate, COMBOSurrogate
from robotorchan.optim import (
    propose_bocs_candidate,
    propose_combo_candidate,
    run_bocs_sequential,
    run_combo_sequential,
)

torch.manual_seed(0)
dtype = torch.double
train_X = torch.tensor([[0., 0.], [1., 0.]], dtype=dtype)

def objective(X):
    return (X[:, :1] + 2 * X[:, 1:2]).to(dtype=dtype)

train_Y = objective(train_X)
print('Initial:', train_X, train_Y, sep='\n')

## 1. BOCS：二次ベイズ回帰

二値入力の一次・交互作用をモデル化します。ここでの実装はガウス係数事前分布を使用し、原著のスパースBOCSを完全再現するものではありません。

In [ ]:
bocs = BOCSSurrogate(train_X, train_Y)
print('BOCS posterior mean:', bocs.posterior(train_X).mean)
next_bocs = propose_bocs_candidate(train_X, train_Y)
print('BOCS candidate:', next_bocs, 'objective:', objective(next_bocs))

## 2. COMBO：グラフ拡散カーネルGP

各二値変数のグラフ構造をカーネルに反映します。候補提案は小規模二値空間の全列挙UCBであり、原著COMBOの全要素を実装するものではありません。

In [ ]:
combo = COMBOSurrogate(train_X, train_Y)
combo.fit()
print('COMBO posterior mean:', combo.posterior(train_X).mean)
next_combo = propose_combo_candidate(train_X, train_Y)
print('COMBO candidate:', next_combo, 'objective:', objective(next_combo))

## 3. 逐次BO：提案 → 評価 → 追加

同一初期データからそれぞれ2回だけ探索します。`objective` は `[1, d]` を受け取り `[1, 1]` を返します。

In [ ]:
bocs_X, bocs_Y = run_bocs_sequential(train_X, train_Y, objective, steps=2)
combo_X, combo_Y = run_combo_sequential(train_X, train_Y, objective, steps=2)
print('BOCS:', bocs_X, bocs_Y.flatten(), sep='\n')
print('COMBO:', combo_X, combo_Y.flatten(), sep='\n')
assert torch.unique(bocs_X, dim=0).shape[0] == 4
assert torch.unique(combo_X, dim=0).shape[0] == 4

## 4. 比較と制限

- 両方式は二値・単一目的・q=1の例です。
- Hamming Local SearchとDiscrete Trust Regionは獲得関数の候補探索を担当します。状態管理は別途必要です。
- Mixedや制約の対応は手法ごとに異なります。
- 複数シードの比較は `python examples/combinatorial_bo_benchmark.py --seeds 3 --steps 5` を実行してください。
- [理論](../../docs/theory/27_combinatorial_bayesian_optimization.md) / [実装ガイド](../../docs/guide/27_combinatorial_bayesian_optimization.md)